# Day 10 — Final Supporting Data Validation

## Objective

Perform final supporting data-quality checks on the cleaned Superstore dataset.

The validation covers:
1. Categorical field quality.
2. Category and Sub-Category relationships.
3. Date integrity and shipping chronology.
4. Identifier integrity.
5. Numerical and categorical business rules.
6. Order-level consistency.
7. Source-level anomaly verification.
8. Return-data availability.
9. Consolidated validation summary.

**Source dataset:** `data/raw/sample_-_superstore.xls`

**Processed dataset:** `data/processed/cleaned_superstore.csv`

The original source and cleaned dataset are treated as read-only inputs. No source-level anomalies are silently corrected.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
SOURCE_PATH = PROJECT_ROOT / "data/raw/sample_-_superstore.xls"
CLEANED_PATH = PROJECT_ROOT / "data/processed/cleaned_superstore.csv"

df = pd.read_csv(CLEANED_PATH, dtype={"Postal Code": "string"})

print("=" * 80)
print("DAY 10 — DATASET OVERVIEW")
print("=" * 80)
print("Dataset:", CLEANED_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing cells:", int(df.isna().sum().sum()))
print("\nColumns:")
print(df.columns.tolist())

assert len(df) == 10194
assert len(df.columns) == 21
assert df.isna().sum().sum() == 0

DAY 10 — DATASET OVERVIEW
Dataset: /Users/hemanth/Sales-Business-Performance-Analytics-Dashboard/data/processed/cleaned_superstore.csv
Rows: 10194
Columns: 21
Missing cells: 0

Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [2]:
print("=" * 80)
print("CATEGORICAL FIELD VALIDATION")
print("=" * 80)

categorical_fields = [
    "Ship Mode", "Segment", "Country/Region",
    "Region", "Category", "Sub-Category"
]

categorical_results = {}

for column in categorical_fields:
    missing = int(df[column].isna().sum())
    whitespace = int(
        df[column].astype("string").str.strip().eq("").sum()
    )
    categorical_results[column] = (
        missing == 0 and whitespace == 0
    )

    print(f"\n{column}")
    print("Unique values:", df[column].nunique())
    print("Missing values:", missing)
    print("Whitespace-only values:", whitespace)
    print(df[column].value_counts().to_string())

print("\nCategorical field checks:",
      sum(categorical_results.values()), "/", len(categorical_results))

expected_mapping = {
    "Furniture": {"Bookcases", "Chairs", "Furnishings", "Tables"},
    "Office Supplies": {
        "Appliances", "Art", "Binders", "Envelopes",
        "Fasteners", "Labels", "Paper", "Storage", "Supplies"
    },
    "Technology": {"Accessories", "Copiers", "Machines", "Phones"}
}

actual_mapping = (
    df.groupby("Category")["Sub-Category"]
    .unique().apply(set).to_dict()
)

mapping_errors = []

for category, expected in expected_mapping.items():
    observed = actual_mapping.get(category, set())
    missing = expected - observed
    unexpected = observed - expected
    print(f"\n{category}: expected={len(expected)}, observed={len(observed)}")
    print("Missing:", sorted(missing) if missing else "None")
    print("Unexpected:", sorted(unexpected) if unexpected else "None")
    if missing or unexpected:
        mapping_errors.append(category)

unexpected_categories = set(df["Category"].unique()) - set(expected_mapping)
invalid_mapping_rows = sum(
    subcategory not in expected_mapping.get(category, set())
    for category, subcategory in zip(df["Category"], df["Sub-Category"])
)

mapping_pass = (
    not mapping_errors
    and not unexpected_categories
    and invalid_mapping_rows == 0
)

print("\nInvalid mapping rows:", invalid_mapping_rows)
print("Unexpected categories:", sorted(unexpected_categories))
print("CATEGORY MAPPING:", "PASS" if mapping_pass else "FAIL")

CATEGORICAL FIELD VALIDATION

Ship Mode
Unique values: 4
Missing values: 0
Whitespace-only values: 0
Ship Mode
Standard Class    6120
Second Class      1979
First Class       1548
Same Day           547

Segment
Unique values: 3
Missing values: 0
Whitespace-only values: 0
Segment
Consumer       5281
Corporate      3090
Home Office    1823

Country/Region
Unique values: 2
Missing values: 0
Whitespace-only values: 0
Country/Region
United States    9994
Canada            200

Region
Unique values: 4
Missing values: 0
Whitespace-only values: 0
Region
West       3253
East       2986
Central    2335
South      1620

Category
Unique values: 3
Missing values: 0
Whitespace-only values: 0
Category
Office Supplies    6128
Furniture          2201
Technology         1865

Sub-Category
Unique values: 17
Missing values: 0
Whitespace-only values: 0
Sub-Category
Binders        1548
Paper          1384
Furnishings    1009
Phones          903
Storage         856
Art             821
Accessories     775
Ch

In [3]:
print("=" * 80)
print("DATE INTEGRITY VALIDATION")
print("=" * 80)

df["Order Date Parsed"] = pd.to_datetime(
    df["Order Date"], format="mixed", errors="coerce"
)
df["Ship Date Parsed"] = pd.to_datetime(
    df["Ship Date"], format="mixed", errors="coerce"
)

invalid_order_dates = int(df["Order Date Parsed"].isna().sum())
invalid_ship_dates = int(df["Ship Date Parsed"].isna().sum())
ship_before_order = int(
    (df["Ship Date Parsed"] < df["Order Date Parsed"]).sum()
)

same_day_shipping = int(
    (df["Ship Date Parsed"] == df["Order Date Parsed"]).sum()
)

shipping_days = (
    df["Ship Date Parsed"] - df["Order Date Parsed"]
).dt.days

print("Invalid Order Dates:", invalid_order_dates)
print("Invalid Ship Dates:", invalid_ship_dates)
print("Order Date range:", df["Order Date Parsed"].min(),
      "to", df["Order Date Parsed"].max())
print("Ship Date range:", df["Ship Date Parsed"].min(),
      "to", df["Ship Date Parsed"].max())
print("Ship Date before Order Date:", ship_before_order)
print("Same-day shipping records:", same_day_shipping)
print("Minimum shipping duration:", shipping_days.min())
print("Maximum shipping duration:", shipping_days.max())
print("\nShipping duration distribution:")
print(shipping_days.value_counts().sort_index().to_string())

date_pass = (
    invalid_order_dates == 0
    and invalid_ship_dates == 0
    and ship_before_order == 0
)
print("\nDATE INTEGRITY:", "PASS" if date_pass else "FAIL")

DATE INTEGRITY VALIDATION
Invalid Order Dates: 0
Invalid Ship Dates: 0
Order Date range: 2023-01-03 00:00:00 to 2026-12-30 00:00:00
Ship Date range: 2023-01-07 00:00:00 to 2027-01-05 00:00:00
Ship Date before Order Date: 0
Same-day shipping records: 523
Minimum shipping duration: 0
Maximum shipping duration: 11

Shipping duration distribution:
0      523
1      369
2     1361
3     1025
4     2838
5     2243
6     1204
7      629
11       2

DATE INTEGRITY: PASS


In [4]:
print("=" * 80)
print("IDENTIFIER INTEGRITY VALIDATION")
print("=" * 80)

identifier_fields = ["Row ID", "Order ID", "Customer ID", "Product ID"]
identifier_results = {}

for column in identifier_fields:
    missing = int(df[column].isna().sum())
    unique = int(df[column].nunique())
    duplicates = int(df[column].duplicated().sum())
    blanks = int(df[column].astype("string").str.strip().eq("").sum())

    identifier_results[column] = (
        missing == 0 and blanks == 0
        and (column != "Row ID" or duplicates == 0)
    )

    print(f"\n{column}")
    print("Missing:", missing)
    print("Unique:", unique)
    print("Duplicate rows:", duplicates)
    print("Blank strings:", blanks)

row_id_pass = (
    df["Row ID"].nunique() == len(df)
    and df["Row ID"].min() == 1
    and df["Row ID"].max() == len(df)
)

order_sizes = df.groupby("Order ID").size()

print("\nRow ID continuous sequence:", row_id_pass)
print("Total rows:", len(df))
print("Unique orders:", df["Order ID"].nunique())
print("Minimum rows per order:", order_sizes.min())
print("Maximum rows per order:", order_sizes.max())

identifier_pass = all(identifier_results.values()) and row_id_pass
print("\nIDENTIFIER INTEGRITY:",
      "PASS" if identifier_pass else "FAIL")

IDENTIFIER INTEGRITY VALIDATION

Row ID
Missing: 0
Unique: 10194
Duplicate rows: 0
Blank strings: 0

Order ID
Missing: 0
Unique: 5111
Duplicate rows: 5083
Blank strings: 0

Customer ID
Missing: 0
Unique: 804
Duplicate rows: 9390
Blank strings: 0

Product ID
Missing: 0
Unique: 1862
Duplicate rows: 8332
Blank strings: 0

Row ID continuous sequence: True
Total rows: 10194
Unique orders: 5111
Minimum rows per order: 1
Maximum rows per order: 14

IDENTIFIER INTEGRITY: PASS


In [5]:
print("=" * 80)
print("BUSINESS RULE VALIDATION")
print("=" * 80)

business_checks = {
    "Sales greater than zero": (df["Sales"] > 0).all(),
    "Quantity is positive": (df["Quantity"] > 0).all(),
    "Quantity contains whole numbers": (df["Quantity"] % 1 == 0).all(),
    "Discount between 0 and 1": df["Discount"].between(0, 1).all(),
    "Profit contains valid numeric values":
        pd.to_numeric(df["Profit"], errors="coerce").notna().all(),
    "Postal Code has no blank strings":
        df["Postal Code"].str.strip().ne("").all(),
    "Ship Mode belongs to expected categories":
        set(df["Ship Mode"].unique()) ==
        {"Standard Class", "Second Class", "First Class", "Same Day"},
    "Segment belongs to expected categories":
        set(df["Segment"].unique()) ==
        {"Consumer", "Corporate", "Home Office"}
}

for name, passed in business_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nNumerical boundaries:")
for column in ["Sales", "Quantity", "Discount", "Profit"]:
    print(f"{column}: min={df[column].min()}, max={df[column].max()}")

print("\nNegative-profit rows:", int((df["Profit"] < 0).sum()))
print("Zero-profit rows:", int((df["Profit"] == 0).sum()))
print("Positive-profit rows:", int((df["Profit"] > 0).sum()))
print("Zero-discount rows:", int((df["Discount"] == 0).sum()))
print("Maximum discount:", df["Discount"].max())

business_pass = all(business_checks.values())
print("\nBUSINESS RULE CHECKS:",
      sum(business_checks.values()), "/", len(business_checks))

BUSINESS RULE VALIDATION
Sales greater than zero: PASS
Quantity is positive: PASS
Quantity contains whole numbers: PASS
Discount between 0 and 1: PASS
Profit contains valid numeric values: PASS
Postal Code has no blank strings: PASS
Ship Mode belongs to expected categories: PASS
Segment belongs to expected categories: PASS

Numerical boundaries:
Sales: min=0.4439999999999999, max=22638.48
Quantity: min=1, max=14
Discount: min=0.0, max=0.8
Profit: min=-6599.978000000001, max=8399.975999999999

Negative-profit rows: 1901
Zero-profit rows: 65
Positive-profit rows: 8228
Zero-discount rows: 4925
Maximum discount: 0.8

BUSINESS RULE CHECKS: 8 / 8


In [6]:
print("=" * 80)
print("ORDER-LEVEL CONSISTENCY VALIDATION")
print("=" * 80)

order_fields = [
    "Customer ID", "Customer Name", "Order Date", "Ship Date",
    "Ship Mode", "Segment", "Country/Region", "City",
    "State/Province", "Postal Code", "Region"
]

order_inconsistencies = {}

for column in order_fields:
    counts = df.groupby("Order ID")[column].nunique(dropna=False)
    inconsistent = counts[counts > 1]
    order_inconsistencies[column] = inconsistent

    print(
        f"{column}:",
        "PASS" if inconsistent.empty
        else f"REVIEW ({len(inconsistent)} orders)"
    )

print("\nInconsistent order IDs by field:")
for column, values in order_inconsistencies.items():
    if not values.empty:
        print(column, ":", values.index.tolist())

total_field_order_inconsistencies = sum(
    len(values) for values in order_inconsistencies.values()
)

print("\nTotal field/order inconsistencies:",
      total_field_order_inconsistencies)
print("Rows per order — minimum:", df.groupby("Order ID").size().min())
print("Rows per order — maximum:", df.groupby("Order ID").size().max())

# Verify identified anomalies against the original source.
affected_orders = [
    "CA-2025-121465", "CA-2023-131807",
    "CA-2026-130494", "CA-2026-131807"
]

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"Source dataset not found: {SOURCE_PATH}")

source = pd.read_excel(SOURCE_PATH)

comparison_fields = [
    "Row ID", "Order ID", "Customer ID", "Customer Name",
    "City", "State/Province", "Postal Code", "Country/Region",
    "Product ID", "Product Name"
]

source_subset = source.loc[
    source["Order ID"].isin(affected_orders), comparison_fields
].copy()

cleaned_subset = df.loc[
    df["Order ID"].isin(affected_orders), comparison_fields
].copy()

source_subset = source_subset.sort_values("Row ID").reset_index(drop=True)
cleaned_subset = cleaned_subset.sort_values("Row ID").reset_index(drop=True)

source_subset = source_subset.astype("string").fillna("<NA>")
cleaned_subset = cleaned_subset.astype("string").fillna("<NA>")

source_anomalies_match = (
    source_subset.equals(cleaned_subset)
    and set(source_subset["Order ID"]) == set(affected_orders)
)

print("\nAffected orders compared with original source:")
print("Affected orders:", affected_orders)
print("Source rows:", len(source_subset))
print("Cleaned rows:", len(cleaned_subset))
print("Source and cleaned values identical:",
      source_anomalies_match)

print("\nSOURCE ANOMALY VERIFICATION:",
      "PASS — anomalies inherited from source"
      if source_anomalies_match else "REVIEW REQUIRED")

ORDER-LEVEL CONSISTENCY VALIDATION
Customer ID: REVIEW (2 orders)
Customer Name: PASS
Order Date: PASS
Ship Date: PASS
Ship Mode: PASS
Segment: PASS
Country/Region: PASS
City: REVIEW (2 orders)
State/Province: PASS
Postal Code: REVIEW (2 orders)
Region: PASS

Inconsistent order IDs by field:
Customer ID : ['CA-2025-121465', 'CA-2026-130494']
City : ['CA-2023-131807', 'CA-2026-131807']
Postal Code : ['CA-2023-131807', 'CA-2026-131807']

Total field/order inconsistencies: 6
Rows per order — minimum: 1
Rows per order — maximum: 14



Affected orders compared with original source:
Affected orders: ['CA-2025-121465', 'CA-2023-131807', 'CA-2026-130494', 'CA-2026-131807']
Source rows: 20
Cleaned rows: 20
Source and cleaned values identical: True

SOURCE ANOMALY VERIFICATION: PASS — anomalies inherited from source


In [7]:
print("=" * 80)
print("RETURN DATA AVAILABILITY")
print("=" * 80)

return_keywords = ["return", "returned", "refund"]
return_columns = [
    column for column in df.columns
    if any(keyword in column.lower() for keyword in return_keywords)
]

print("Return-related columns:",
      return_columns if return_columns else "None found")

return_validation_applicable = len(return_columns) > 0
print(
    "RETURN VALIDATION:",
    "POSSIBLE" if return_validation_applicable
    else "NOT APPLICABLE — NO RETURN FIELDS"
)

RETURN DATA AVAILABILITY
Return-related columns: None found
RETURN VALIDATION: NOT APPLICABLE — NO RETURN FIELDS


In [8]:
print("=" * 80)
print("FINAL DAY 10 VALIDATION SUMMARY")
print("=" * 80)

summary = {
    "Categorical fields": all(categorical_results.values()),
    "Category/Sub-Category mapping": mapping_pass,
    "Date integrity": date_pass,
    "Identifier integrity": identifier_pass,
    "Business rules": business_pass,
    "Source anomaly verification": source_anomalies_match
}

for name, passed in summary.items():
    print(f"{name}: {'PASS' if passed else 'FAIL / REVIEW'}")

print("Order-level inconsistencies:",
      total_field_order_inconsistencies,
      "(source-level anomalies; retained unchanged)")
print("Return validation:",
      "APPLICABLE" if return_validation_applicable
      else "NOT APPLICABLE — NO RETURN FIELDS")

print("\nPassed validation groups:",
      sum(summary.values()), "/", len(summary))
print("Dataset rows:", len(df))
print("Dataset columns:", len(pd.read_csv(CLEANED_PATH, nrows=0).columns))
print("Unique orders:", df["Order ID"].nunique())

print("\nFINAL INTERPRETATION")
print("Supporting checks passed.")
print("Known source-level order anomalies were retained unchanged.")
print("Return analysis is not possible with the available fields.")
print("=" * 80)

FINAL DAY 10 VALIDATION SUMMARY
Categorical fields: PASS
Category/Sub-Category mapping: PASS
Date integrity: PASS
Identifier integrity: PASS
Business rules: PASS
Source anomaly verification: PASS
Order-level inconsistencies: 6 (source-level anomalies; retained unchanged)
Return validation: NOT APPLICABLE — NO RETURN FIELDS

Passed validation groups: 6 / 6
Dataset rows: 10194
Dataset columns: 21
Unique orders: 5111

FINAL INTERPRETATION
Supporting checks passed.
Known source-level order anomalies were retained unchanged.
Return analysis is not possible with the available fields.
